In [22]:
import glob
import os
import pandas as pd

all_files = glob.glob('../data/raw/*.csv')

data_dict={}
for file in all_files:
    file_name = os.path.splitext(os.path.basename(file))[0]
    data_dict[file_name] = pd.read_csv(file)

print((data_dict.keys()))


dict_keys(['indication', 'generic', 'manufacturer', 'dosage_form', 'drug_class', 'medicine'])


In [23]:
data_dict['medicine'].head()

,brand id,brand name,type,slug,dosage form,generic,strength,manufacturer,package container,Package Size
0,4077,A-Cold,allopathic,a-coldsyrup4-mg5-ml,Syrup,Bromhexine Hydrochloride,4 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 40.12,NaN
1,4006,A-Cof,allopathic,a-cofsyrup10-mg30-mg125-mg5-ml,Syrup,Dextromethorphan + Pseudoephedrine + Triprolidine,(10 mg+30 mg+1.25 mg)/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 100.00,NaN
2,6174,A-Clox,allopathic,a-cloxinjection500-mgvial,Injection,Cloxacillin Sodium,500 mg/vial,ACME Laboratories Ltd.,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),",(5's pack: ৳ 142.15)
3,6173,A-Clox,allopathic,a-cloxinjection250-mgvial,Injection,Cloxacillin Sodium,250 mg/vial,ACME Laboratories Ltd.,"250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),",(5's pack: ৳ 100.00)
4,6172,A-Clox,allopathic,a-cloxpowder-for-suspension125-mg5-ml,Powder for Suspension,Cloxacillin Sodium,125 mg/5 ml,ACME Laboratories Ltd.,100 ml bottle: ৳ 45.00,NaN


### Standardize Column Names

In [24]:
for name, df in data_dict.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f"{name}: {list(df.columns)}")

indication: ['indication_id', 'indication_name', 'slug', 'generics_count']
generic: ['generic_id', 'generic_name', 'slug', 'monograph_link', 'drug_class', 'indication', 'indication_description', 'therapeutic_class_description', 'pharmacology_description', 'dosage_description', 'administration_description', 'interaction_description', 'contraindications_description', 'side_effects_description', 'pregnancy_and_lactation_description', 'precautions_description', 'pediatric_usage_description', 'overdose_effects_description', 'duration_of_treatment_description', 'reconstitution_description', 'storage_conditions_description', 'descriptions_count']
manufacturer: ['manufacturer_id', 'manufacturer_name', 'slug', 'generics_count', 'brand_names_count']
dosage_form: ['dosage_form_id', 'dosage_form_name', 'slug', 'brand_names_count']
drug_class: ['drug_class_id', 'drug_class_name', 'slug', 'generics_count']
medicine: ['brand_id', 'brand_name', 'type', 'slug', 'dosage_form', 'generic', 'strength', 'ma

### Inspect Missing Values in Medicine Table

In [25]:
medicine = data_dict['medicine'].copy()

print(f"Total rows: {len(medicine)}")
print(f"Duplicate rows: {medicine.duplicated().sum()}\n")
print("Missing values per column:")
print(medicine.isnull().sum())

Total rows: 21714
Duplicate rows: 0

Missing values per column:
brand_id                0
brand_name              0
type                    0
slug                    0
dosage_form             0
generic                 2
strength              849
manufacturer            0
package_container      42
package_size         7779
dtype: int64


### Handle Missing Values and Drop Redundant Columns
1. `generic` (2 nulls dropped): Dropping 2 rows has zero impact on market totals.
2. `strength` (849 nulls filled as 'Not Specified'): These are valid multi-ingredient vitamins and herbal syrups where a single mg strength does not apply. Dropping them would falsely shrink manufacturer portfolios.
3. `package_size` (7,779 nulls, column dropped): Redundant, because the exact pack size text is already embedded inside `package_container` column.
4. `package_container` (42 nulls kept): Kept as `NaN` so these registered brands still count toward manufacturer portfolio size. They will be excluded from price calculations.

In [26]:
medicine = medicine.dropna(subset=['generic']).copy() #1.

medicine['strength'] = medicine['strength'].fillna('Not Specified') #2.

medicine = medicine.drop(columns=['package_size']) #3.


print(f"Remaining rows: {len(medicine)}")
print("\nUpdated missing values:")
print(medicine.isnull().sum())

Remaining rows: 21712

Updated missing values:
brand_id              0
brand_name            0
type                  0
slug                  0
dosage_form           0
generic               0
strength              0
manufacturer          0
package_container    42
dtype: int64


### Extract price from package_container column
* Extract the first numeric value following the `৳` symbol as the price (`price_bdt`).
* Rows without a `৳` symbol evaluate to `NaN`.

In [40]:
medicine[['brand_name', 'dosage_form', 'package_container']].head()

,brand_name,dosage_form,package_container
0,A-Cold,Syrup,100 ml bottle: ৳ 40.12
1,A-Cof,Syrup,100 ml bottle: ৳ 100.00
2,A-Clox,Injection,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),"
3,A-Clox,Injection,"250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),"
4,A-Clox,Powder for Suspension,100 ml bottle: ৳ 45.00


In [39]:
medicine['price_bdt'] = (
    medicine['package_container']
    .str.extract(r'৳\s*([\d,]+(?:\.\d+)?)')[0]
    .str.replace(',', '', regex=False)
    .astype(float)
)


print(f"Valid prices extracted: {medicine['price_bdt'].notna().sum()}")
print(f"Missing prices: {medicine['price_bdt'].isna().sum()}")

medicine[['brand_name', 'dosage_form', 'package_container', 'price_bdt']].head(10)

Valid prices extracted: 21634
Missing prices: 78


,brand_name,dosage_form,package_container,price_bdt
0,A-Cold,Syrup,100 ml bottle: ৳ 40.12,40.12
1,A-Cof,Syrup,100 ml bottle: ৳ 100.00,100.00
2,A-Clox,Injection,"500 mg vial: ৳ 28.43,(5's pack: ৳ 142.15),",28.43
3,A-Clox,Injection,"250 mg vial: ৳ 20.00,(5's pack: ৳ 100.00),",20.00
4,A-Clox,Powder for Suspension,100 ml bottle: ৳ 45.00,45.00
5,A-Clox,Capsule,"Unit Price: ৳ 5.98,(100's pack: ৳ 598.00),",5.98
6,A-Care,Tablet,"Unit Price: ৳ 2.50,(20's pack: ৳ 50.00),",2.50
7,A-Card,Tablet,"Unit Price: ৳ 1.43,(100's pack: ৳ 143.00),",1.43
8,A-Calm,Tablet,"Unit Price: ৳ 6.00,(50's pack: ৳ 300.00),",6.00
9,A-Cal DX,Tablet,"Unit Price: ৳ 7.50,(30's pack: ৳ 225.00),",7.50


### Classify Pricing Basis (per_unit vs. per_container)
* `per_unit`: Rows starting with "Unit Price:" (individual tablets, capsules, suppositories).
* `per_container`: Liquids, injectables, creams, and inhalers priced by bottle, vial or tube.
* `missing`: NaN prices.

In [45]:
import numpy as np


conditions = [
    medicine['price_bdt'].isna(),
    medicine['package_container'].str.startswith('Unit Price:', na=False)
]
choices = ['missing', 'per_unit']

medicine['pricing_basis'] = np.select(conditions, choices, default='per_container')


print(medicine['pricing_basis'].value_counts())

pricing_basis
per_unit         13500
per_container     8134
missing             78
Name: count, dtype: int64


### Checking Extracted Prices for Anomalies
* Verify summary statistics across per_unit and per_container.
* Inspect the highest-priced rows to confirm whether extreme values are real specialty medicines or parsing errors.
##### Insight
* Extremely right skewed.
* When analyzing we must use Median and IQR

In [62]:
print(medicine.groupby('pricing_basis')['price_bdt'].describe().round(2))


medicine.sort_values(by='price_bdt', ascending=False)[
    ['brand_name', 'dosage_form', 'generic', 'package_container', 'price_bdt']
].head(10)


                 count    mean      std   min    25%    50%     75%        max
pricing_basis                                                                 
missing            0.0     NaN      NaN   NaN    NaN    NaN     NaN        NaN
per_container   8134.0  866.04  7530.74  2.47  40.00  80.55  200.00  340000.00
per_unit       13500.0   30.70   177.44  0.09   4.01   8.00   15.05    8976.66


,brand_name,dosage_form,generic,package_container,price_bdt
18971,Tecentriq,IV Infusion,Atezolizumab,"1200 mg vial: ৳ 340,000.00",340000.00
15278,Pemro,IV Infusion,Pembrolizumab,"100 mg vial: ৳ 195,000.00",195000.00
15388,Perjeta,IV Infusion,Pertuzumab,"14 ml vial: ৳ 175,000.00",175000.00
10261,Kadcyla,IV Infusion,Trastuzumab Emtansine,"160 mg vial: ৳ 150,500.00",150500.00
13636,Nivomab,IV Infusion,Nivolumab,"100 mg vial: ৳ 146,000.00",146000.00
10229,Jevtana,IV Infusion,Cabazitaxel,"1.5 ml vial: ৳ 142,687.22",142687.22
20687,Visudyne,Injection,Verteporfin,"15 mg vial: ৳ 137,799.79",137799.79
9030,Gazyva,Injection,Obinutuzumab,"1000 mg vial: ৳ 112,000.00",112000.00
17265,Ronapreve,Injection,Casirivimab + Imdevimab,"1332 mg vial & 1332 mg vial: ৳ 106,250.00",106250.00
18079,Simulect,IV Infusion,Basiliximab,"20 mg vial: ৳ 103,273.56",103273.56


### Merge `medicine` table with `generic` table
* Select only generic_id, generic_name, drug_class, indication from `generic` table.
* Fill missing drug_class and indication with "Not Specified".
* Merge onto `medicine` via a left join.

In [80]:
generic = data_dict['generic'].copy()
# generic.head()
# generic.isnull().sum()

generic_clean = generic[['generic_id', 'generic_name', 'drug_class', 'indication']].copy()
# generic_clean.isnull().sum()

generic_clean['drug_class'] = generic_clean['drug_class'].fillna('Not Specified')
generic_clean['indication'] = generic_clean['indication'].fillna('Not Specified')

# # Left join medicine with generic_clean
medicine_generic_join = medicine.merge(
    generic_clean,
    left_on='generic',
    right_on='generic_name',
    how='left'
)
#dropping duplicate column from right table
medicine_generic_join = medicine_generic_join.drop(columns=['generic_name'])


print(f"Rows after merge: {len(medicine_generic_join)}")
print(medicine_generic_join.isnull().sum())

Rows after merge: 21712
brand_id              0
brand_name            0
type                  0
slug                  0
dosage_form           0
generic               0
strength              0
manufacturer          0
package_container    42
price_bdt            78
pricing_basis         0
generic_id            0
drug_class            0
indication            0
dtype: int64


### Identify Unmanufactured Generics

In [111]:
total_generics = generic_clean['generic_name'].unique()
active_generics = medicine_generic_join['generic'].unique()

# unmanufactured_generics = np.setdiff1d(total_generics, active_generics)
unmanufactured_generics = generic_clean[
    ~generic_clean['generic_name'].isin(active_generics)
].copy()


print(f'Total Generics: {len(total_generics)}')
print(f'Active Generics: {len(active_generics)}')
print(f'Unmanufactured Generics: {len(unmanufactured_generics)}')
unmanufactured_generics.head(10)

Total Generics: 1711
Active Generics: 1661
Unmanufactured Generics: 50


,generic_id,generic_name,drug_class,indication
121,80,Artemether,Anti-malarial drugs,Schistosomiasis
219,1276,Benzyl Alcohol,Other scalp preparations,Pediculosis capitis
223,1254,Benzocaine + Camphor + Methanol + Phenol,Topical Local Anesthetics,Toothache
258,1242,Betamethasone + Gentamicin,Other Topical corticosteroids,Psoriasis
307,1397,Butoconazole Nitrate,Drugs used in Vaginal and Vulval condition,Vulvovaginal candidiasis
397,219,Cefpirome Sulphate,Fourth generation Cephalosporins,Susceptible infections
398,1274,Cefoxitin,Second generation Cephalosporins,Urinary tract infection
487,260,Cinchocaine Hydrochloride,Local & Surface anesthesia,Surface anaesthesia
490,258,Cinacalcet,Calcium Regulator,Secondary hyperparathyroidism
645,395,Doxylamine Succinate,Sedating Anti-histamine,Watery eye


### Export Cleaned Datasets
* Save `medicine_generic_join` as `master_medicines.csv` for Exploratory Data Analysis and Dashboard.
* Save `unmanufactured_generics` as `unmanufactured_generics.csv` for white-space opportunity analysis.

In [113]:
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)

medicine_generic_join.to_csv(f'{output_dir}/master_medicines.csv', index=False)
unmanufactured_generics.to_csv(f'{output_dir}/unmanufactured_generics.csv', index=False)

print(os.listdir('../data/processed'))

['unmanufactured_generics.csv', 'master_medicines.csv']
